# 02 - Data validation and preparation

This notebook converts the source artifacts admitted by Notebook 01 into validated, source-faithful RR tables. It establishes coverage, schema, uniqueness, missingness, units, empirical ranges, and causal alignment before feature engineering or model fitting.

The notebook is restricted to **1 January 2015 through 31 December 2020**. It does not download data, tune a model, select features using the 2020 evaluation outcome, globally standardise population, impute unexplained values, or introduce any private-repository feature.


## 1. Preparation contract

This stage writes validated AEMO and regional-weather tables, the original annual Queensland population table, and the causal five-minute base. Population is one statewide scalar only; no regional population channels are created.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (candidate, *candidate.parents) if (path / 'environment.yml').is_file() and (path / 'data').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_provenance import audit_manifest, load_yaml, source_inventory
from src.population_pipeline import load_annual_qld_population
from src.rr_data_preparation import (
    REGIONS, STUDY_END, STUDY_START, causal_weather_alignment_audit,
    development_only, prepare_aemo, prepare_weather, write_validated_tables,
)

CONFIG = load_yaml(PROJECT_ROOT / 'config' / 'data_acquisition.yml')
REGISTRY = load_yaml(PROJECT_ROOT / CONFIG['paths']['source_registry_path'])
CONTRACTS = load_yaml(PROJECT_ROOT / 'data' / 'schemas' / 'source_contracts.yml')
MANIFEST_PATH = PROJECT_ROOT / CONFIG['paths']['manifest_path']
sns.set_theme(style='whitegrid', context='notebook')
print(f'RR root: {PROJECT_ROOT}')
print(f'Python:  {sys.executable}')
print(f'Period:  {STUDY_START.date()} to {STUDY_END.date()}')

## 2. Enforce the Notebook 01 acquisition gate

Preparation is refused unless every declared source has its current checksum registered and every manifest record remains inside the RR temporal boundary.

In [ ]:
inventory = source_inventory(PROJECT_ROOT, REGISTRY, MANIFEST_PATH)
manifest_audit = audit_manifest(PROJECT_ROOT, MANIFEST_PATH)
display(inventory)
if not inventory['status'].eq('registered and verified').all():
    raise RuntimeError('Notebook 01 acquisition gate has not passed for every source.')
audit_columns = ['exists', 'size_matches', 'sha256_matches', 'coverage_within_2020']
if manifest_audit.empty or not manifest_audit[audit_columns].all().all():
    raise RuntimeError('Manifest identity or temporal-boundary audit failed.')
display(Markdown('### Acquisition and provenance gate: PASS'))

## 3. Load and validate the temporal source tables

AEMO retains the full physical-record metadata and `DEMANDFORECAST`, but the latter remains an external comparison field and is not admitted as a model input. Weather retains all acquired variables for the five declared regional nodes. Population is reconstructed from the exact historical release vintages rather than from a later revised workbook.


In [ ]:
aemo_path = PROJECT_ROOT / CONFIG['aemo_dispatchregionsum']['canonical_output']
weather_path = PROJECT_ROOT / CONFIG['open_meteo']['canonical_output']
aemo = prepare_aemo(pd.read_parquet(aemo_path))
weather = prepare_weather(pd.read_parquet(weather_path))

source_summary = pd.DataFrame([
    {'table': 'AEMO QLD1', 'rows': len(aemo), 'start': aemo['SETTLEMENTDATE'].min(), 'end': aemo['SETTLEMENTDATE'].max(), 'columns': len(aemo.columns), 'duplicate_keys': int(aemo['SETTLEMENTDATE'].duplicated().sum())},
    {'table': 'Regional weather', 'rows': len(weather), 'start': weather['datetime'].min(), 'end': weather['datetime'].max(), 'columns': len(weather.columns), 'duplicate_keys': int(weather.duplicated(['region', 'datetime']).sum())},
])
display(source_summary)

## 4. Data-quality audit

Missingness is reported, not silently repaired. The AEMO timeline begins at 00:05 on 1 January 2015; this source boundary is preserved rather than inventing a 00:00 record. Hourly weather is required to have one unique record per region and hour.

In [ ]:
expected_aemo = pd.date_range(aemo['SETTLEMENTDATE'].min(), aemo['SETTLEMENTDATE'].max(), freq='5min')
aemo_gaps = expected_aemo.difference(aemo['SETTLEMENTDATE'])
quality = pd.DataFrame({
    'measure': ['AEMO missing expected intervals', 'AEMO duplicate timestamps', 'AEMO missing numeric cells', 'Weather duplicate region-hours', 'Weather missing numeric cells'],
    'value': [len(aemo_gaps), int(aemo['SETTLEMENTDATE'].duplicated().sum()), int(aemo.select_dtypes(include='number').isna().sum().sum()), int(weather.duplicated(['region', 'datetime']).sum()), int(weather.select_dtypes(include='number').isna().sum().sum())],
})
display(quality)
weather_missing = weather.isna().sum().rename('missing').to_frame()
weather_missing['percent'] = weather_missing['missing'] / len(weather) * 100
display(weather_missing.loc[weather_missing['missing'].gt(0)].sort_values('percent', ascending=False))

## 5. Initial demand exploration

These descriptive summaries use **only the 2015–2019 development period**. The fixed 2020 chronological evaluation year is excluded from every EDA statistic and plot and is not used to choose features or hyperparameters.

In [ ]:
development_aemo = development_only(aemo, 'SETTLEMENTDATE')
if development_aemo['SETTLEMENTDATE'].dt.year.max() != 2019:
    raise RuntimeError('Development EDA boundary is not locked at 2019.')
print(f'EDA period: {development_aemo["SETTLEMENTDATE"].min()} to {development_aemo["SETTLEMENTDATE"].max()}')
demand_summary = development_aemo['TOTALDEMAND'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename('TOTALDEMAND_MW').to_frame()
display(demand_summary)
annual = development_aemo.assign(year=development_aemo['SETTLEMENTDATE'].dt.year).groupby('year')['TOTALDEMAND'].agg(['count', 'mean', 'std', 'min', 'median', 'max'])
display(annual)

daily = development_aemo.set_index('SETTLEMENTDATE')['TOTALDEMAND'].resample('D').mean()
fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
daily.plot(ax=axes[0], linewidth=0.8, color='#176B87', title='Queensland daily mean operational demand, 2015–2019')
axes[0].set_ylabel('MW')
profile = development_aemo.assign(hour=development_aemo['SETTLEMENTDATE'].dt.hour + development_aemo['SETTLEMENTDATE'].dt.minute / 60).groupby('hour')['TOTALDEMAND'].agg(['mean', 'median'])
profile.plot(ax=axes[1], color=['#176B87', '#E8751A'], title='Five-minute demand profile by time of day, 2015–2019')
axes[1].set(xlabel='Hour of day', ylabel='MW')
plt.show()

## 6. Regional weather ranges

Ranges are retained in provider units and are calculated only from **2015–2019**. The locked 2020 weather observations are not included in these summaries or plots. Scaling belongs to the later feature/training contract and must be fitted on training data only.

In [ ]:
development_weather = development_only(weather, 'datetime')
if development_weather['datetime'].dt.year.max() != 2019:
    raise RuntimeError('Development weather EDA boundary is not locked at 2019.')
weather_numeric = [column for column in development_weather.select_dtypes(include='number').columns if column not in {'latitude', 'longitude'}]
weather_ranges = development_weather.groupby('region')[weather_numeric].agg(['min', 'max'])
display(weather_ranges)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
development_weather.groupby('region')['temperature_2m'].mean().sort_values().plot.barh(ax=axes[0], color='#176B87', title='Mean temperature by region, 2015–2019')
development_weather.groupby('region')['relative_humidity_2m'].mean().sort_values().plot.barh(ax=axes[1], color='#E8751A', title='Mean relative humidity by region, 2015–2019')
axes[0].set_xlabel('°C'); axes[1].set_xlabel('%')
plt.show()

## 7. Causal hourly-to-five-minute alignment audit

For every forecast origin, the permitted weather record is the most recent timestamp at or before that origin. Forward interpolation and nearest-neighbour matching are prohibited because they could select a future hourly value. The audit is performed across the full five-minute timeline without materialising a redundant wide weather table.

In [ ]:
alignment_audit = causal_weather_alignment_audit(aemo['SETTLEMENTDATE'], weather)
display(alignment_audit)
if alignment_audit['future_weather_rows'].ne(0).any():
    raise RuntimeError('Causal weather alignment selected a future observation.')
if alignment_audit['maximum_age_minutes'].gt(55).any():
    raise RuntimeError('Unexpected weather age after backward hourly alignment.')

## 8. Validate the annual Queensland population series

The `TCN_starNLL` contract uses one annual statewide population scalar at late fusion. Each reference-year value becomes active on 1 April of the following year.


In [ ]:
annual_population = load_annual_qld_population(PROJECT_ROOT)
display(annual_population)
if annual_population['reference_year'].tolist() != list(range(2013, 2021)):
    raise RuntimeError('Annual Queensland population years are incomplete.')


## 9. Write validated RR tables

The outputs remain pre-feature and unscaled. `DEMANDFORECAST` is preserved only for later external benchmarking. The annual statewide population values remain raw so each training fold can fit its own standardizer.


In [ ]:
output_paths = write_validated_tables(PROJECT_ROOT, aemo, weather)
for label, path in output_paths.items():
    print(f'{label}: {path.relative_to(PROJECT_ROOT)}')


## 10. Completion gate

Notebook 02 is complete only when the validated tables remain inside 2015-2020, weather alignment is backward-only, and the annual population join never admits a future value.


In [ ]:
base = pd.read_parquet(output_paths['base'])
checks = {
    'all_sources_verified': bool(inventory['status'].eq('registered and verified').all()),
    'weather_alignment_backward_only': bool(alignment_audit['future_weather_rows'].eq(0).all()),
    'population_complete': bool(base['total_qld_population'].notna().all()),
    'population_contract_exact': list(base.filter(like='population').columns) == ['total_qld_population'],
    'base_ends_in_2020': pd.to_datetime(base['SETTLEMENTDATE']).max().year == 2020,
}
display(pd.Series(checks, name='pass'))
if not all(checks.values()):
    raise RuntimeError('Notebook 02 completion gate failed.')


## 11. Next stage

Notebook 03 freezes the public feature and evaluation contracts, including training-fold-only standardisation of the one annual statewide population scalar.


## 12. TCN_starNLL temporal-source validation

The SOI and regional-capacity tables are validated independently before model
assembly. The SOI availability timestamp must never exceed the forecast
origin. Capacity is a monthly cumulative stock derived only from installations
recorded by the applicable frozen CER artifact. Both sources end no later than
31 December 2020.


In [ ]:
from src.tcn_star_nll_sources import build_publication_safe_soi

capacity = pd.read_parquet(PROJECT_ROOT / 'data/processed/SGU_Solar_monthly_2015_2020.parquet')
soi_origins = pd.date_range('2015-01-01', '2020-12-31 23:55', freq='5min')
soi = build_publication_safe_soi(
    soi_origins,
    PROJECT_ROOT / 'data/raw/bom/soi_monthly_through_2020.csv',
)
assert soi.isna().sum().sum() == 0
assert len(capacity) == 5 * 72
assert soi.index.max() <= pd.Timestamp('2020-12-31 23:55')
assert capacity['Date'].max() <= pd.Timestamp('2020-12-31')
{'soi_rows': len(soi), 'capacity_rows': len(capacity), 'status': 'PASS'}
